# 00 — Environment Smoke Test
Verify Colab GPU, ManiSkill, and Hydra config loading before running any experiments.

In [ ]:
# Cell 1: Mount Google Drive (Colab only)
import os
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print('Drive mounted.')
    DRIVE_ROOT = '/content/drive/MyDrive/vla-risk-wrapper'
    os.environ['DRIVE_ROOT'] = DRIVE_ROOT
    os.makedirs(f'{DRIVE_ROOT}/data', exist_ok=True)
    os.makedirs(f'{DRIVE_ROOT}/checkpoints', exist_ok=True)
    os.makedirs(f'{DRIVE_ROOT}/results', exist_ok=True)
    os.makedirs(f'{DRIVE_ROOT}/figures', exist_ok=True)
except ImportError:
    print('Not running in Colab — skipping Drive mount.')

In [ ]:
# Cell 2: GPU check
import torch
cuda_available = torch.cuda.is_available()
if cuda_available:
    print('CUDA available:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')
else:
    print('WARNING: No CUDA GPU detected. Latency measurements will be CPU-based.')

In [ ]:
# Cell 3: ManiSkill environment test
import mani_skill
import gymnasium as gym

env = gym.make('PickCube-v1', obs_mode='state_dict', render_mode=None)
obs, info = env.reset(seed=0)
print('PickCube-v1 obs keys:', list(obs.keys()))
action = env.action_space.sample()
obs2, reward, done, trunc, info2 = env.step(action)
print('Step succeeded. Reward:', reward)
env.close()
print('ManiSkill OK.')

In [ ]:
# Cell 4: Hydra config loading
from omegaconf import OmegaConf
import os

# Find config relative to notebook location
nb_dir = os.path.dirname(os.path.abspath('00_env_smoke_test.ipynb'))
cfg_path = os.path.join(nb_dir, '..', 'configs', 'base.yaml')

cfg = OmegaConf.load(cfg_path)
print(OmegaConf.to_yaml(cfg))

In [ ]:
# Cell 5: Final check
print('Smoke test passed!')